# 02 - Mushroom classification met Logistic Regression

**Auteur:** Brent Van Gelder  
**Dataset:** `mushroom_project_dataset.csv`  
**Doel:** voorspellen of een paddenstoel eetbaar (`e`) of giftig/onbekend (`p`) is.

> Dit is een onderwijsmodel en mag nooit worden gebruikt om echte paddenstoelen veilig te verklaren.

Deze notebook gebruikt dezelfde datacontroles, uitgesloten ruiskolommen, random seed, 70/15/15-split en evaluatiemetrics als de Random Forest-notebook. Numerieke kenmerken worden na dezelfde imputatie gestandaardiseerd, omdat Logistic Regression gevoelig is voor verschillen in schaal.

## Technologie

Logistic Regression is een lineair en relatief goed uitlegbaar classificatiemodel. Het vormt een nuttige controle: als dit eenvoudige model dicht bij complexere modellen komt, is extra complexiteit mogelijk niet nodig.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 1. Data en vaste instellingen

De meegeleverde project-CSV wordt lokaal geladen. De twee expliciet als `jumbled_noise` benoemde kolommen worden in alle modelnotebooks uitgesloten. We controleren labels, ontbrekende waarden en duplicaten voordat het model wordt gebouwd.

In [ ]:
RANDOM_STATE = 42
DATA_PATH = Path("mushroom_project_dataset.csv")
TARGET = "class"
EDIBLE_LABEL = "e"
POISONOUS_LABEL = "p"
RUN_FINAL_TEST = False

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset niet gevonden: {DATA_PATH.resolve()}")

df = pd.read_csv(DATA_PATH)
if TARGET not in df.columns:
    raise ValueError(f"Doelkolom '{TARGET}' ontbreekt.")
if df[TARGET].isna().any():
    raise ValueError("De doelkolom bevat ontbrekende waarden.")
if set(df[TARGET].unique()) != {EDIBLE_LABEL, POISONOUS_LABEL}:
    raise ValueError(f"Onverwachte labels: {set(df[TARGET].unique())}")

noise_columns = [column for column in df.columns if column.startswith("jumbled_noise_")]
model_df = df.drop(columns=noise_columns).copy()

summary = pd.DataFrame({
    "datatype": model_df.dtypes.astype(str),
    "ontbrekend": model_df.isna().sum(),
    "ontbrekend_percentage": (100 * model_df.isna().mean()).round(1),
    "unieke_waarden": model_df.nunique(dropna=True),
}).sort_values("ontbrekend_percentage", ascending=False)

print(f"Rijen: {len(model_df):,}")
print(f"Exact dubbele rijen: {model_df.duplicated().sum()}")
print(f"Uitgesloten ruiskolommen: {noise_columns}")
display(model_df[TARGET].value_counts().rename_axis("klasse").to_frame("aantal"))
display(summary)

## 2. Dezelfde train-, validatie- en testset

Met `random_state=42` en dezelfde tweestapssplit krijgt iedere notebook exact dezelfde 70% trainingsrijen, 15% validatierijen en 15% testrijen. `stratify` bewaart de klasseverhouding. De testset blijft vergrendeld totdat alle keuzes zijn gemaakt.

In [ ]:
X = model_df.drop(columns=TARGET)
y = model_df[TARGET]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)
X_validation, X_test, y_validation, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

split_summary = pd.DataFrame({
    "rijen": [len(X_train), len(X_validation), len(X_test)],
    "aandeel_dataset": [len(X_train) / len(X), len(X_validation) / len(X), len(X_test) / len(X)],
    "aandeel_giftig": [
        (y_train == POISONOUS_LABEL).mean(),
        (y_validation == POISONOUS_LABEL).mean(),
        (y_test == POISONOUS_LABEL).mean(),
    ],
}, index=["train", "validatie", "test"])
display(split_summary.style.format({
    "aandeel_dataset": "{:.1%}",
    "aandeel_giftig": "{:.1%}",
}))

## 3. Cleaning, preprocessing en model

Numerieke ontbrekende waarden krijgen de trainingsmediaan en een missingness-indicator. Daarna worden numerieke waarden gestandaardiseerd. Categorische ontbrekende waarden worden `__missing__`; one-hot encoding negeert later onbekende categorieën veilig. De volledige preprocessing wordt uitsluitend op de trainingsset geleerd.

In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
    ]), numeric_features),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="__missing__")),
        ("one_hot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), categorical_features),
], verbose_feature_names_out=False)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        max_iter=3000,
        solver="lbfgs",
        random_state=RANDOM_STATE,
    )),
])

model.fit(X_train, y_train)
print("Logistic Regression getraind op de trainingsset.")

## 4. Dezelfde evaluatiemetrics

De giftige klasse `p` is de positieve klasse. F2 weegt recall zwaarder dan precision, omdat een giftige paddenstoel ten onrechte als eetbaar classificeren de ernstigste fout is. We rapporteren daarnaast accuracy, balanced accuracy, F1, ROC-AUC en average precision.

In [ ]:
def evaluate_classifier(fitted_model, X_eval, y_eval, split_name):
    predictions = fitted_model.predict(X_eval)
    poisonous_index = list(fitted_model.classes_).index(POISONOUS_LABEL)
    poisonous_probability = fitted_model.predict_proba(X_eval)[:, poisonous_index]
    y_binary = (y_eval == POISONOUS_LABEL).astype(int)

    metrics = pd.Series({
        "accuracy": accuracy_score(y_eval, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_eval, predictions),
        "precision_giftig": precision_score(
            y_eval, predictions, pos_label=POISONOUS_LABEL, zero_division=0
        ),
        "recall_giftig": recall_score(
            y_eval, predictions, pos_label=POISONOUS_LABEL, zero_division=0
        ),
        "f1_giftig": f1_score(
            y_eval, predictions, pos_label=POISONOUS_LABEL, zero_division=0
        ),
        "f2_giftig": fbeta_score(
            y_eval, predictions, beta=2, pos_label=POISONOUS_LABEL, zero_division=0
        ),
        "roc_auc": roc_auc_score(y_binary, poisonous_probability),
        "average_precision": average_precision_score(y_binary, poisonous_probability),
    }, name=split_name)

    display(metrics.to_frame().T.style.format("{:.3f}"))
    print(classification_report(
        y_eval,
        predictions,
        labels=[EDIBLE_LABEL, POISONOUS_LABEL],
        target_names=["eetbaar", "giftig/onbekend"],
        zero_division=0,
    ))
    ConfusionMatrixDisplay.from_predictions(
        y_eval,
        predictions,
        labels=[EDIBLE_LABEL, POISONOUS_LABEL],
        display_labels=["eetbaar", "giftig/onbekend"],
        cmap="Blues",
    )
    plt.title(f"Confusion matrix - {split_name}")
    plt.grid(False)
    plt.show()
    return metrics

## 5. Validatie

Alle modelbeslissingen worden op de validatieset beoordeeld. De onafhankelijke testset wordt hier niet gebruikt.

In [ ]:
validation_metrics = evaluate_classifier(
    model, X_validation, y_validation, "logistic_regression_validatie"
)

## 6. Coëfficiënten interpreteren

Een positieve coëfficiënt duwt de voorspelling richting de klasse die scikit-learn als tweede klasse bewaart; hieronder controleren we expliciet welke klasse dat is. Grote absolute waarden hebben de meeste invloed, maar tonen geen oorzakelijk verband.

In [ ]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
classifier = model.named_steps["classifier"]
positive_class = classifier.classes_[1]
coefficients = (
    pd.DataFrame({
        "feature": feature_names,
        "coefficient": classifier.coef_[0],
    })
    .assign(abs_coefficient=lambda table: table["coefficient"].abs())
    .sort_values("abs_coefficient", ascending=False)
)

print(f"Positieve coëfficiënten wijzen richting klasse: {positive_class}")
display(coefficients.head(20))
sns_data = coefficients.head(20).sort_values("coefficient")
colors = np.where(sns_data["coefficient"] >= 0, "firebrick", "forestgreen")
plt.figure(figsize=(8, 7))
plt.barh(sns_data["feature"], sns_data["coefficient"], color=colors)
plt.title("Sterkste Logistic Regression-coëfficiënten")
plt.xlabel("Coëfficiënt")
plt.tight_layout()
plt.show()

## 7. Definitieve testset - voorlopig vergrendeld

Zet `RUN_FINAL_TEST` pas op `True` nadat tuning en eventuele drempelkeuzes definitief zijn vastgelegd.

In [ ]:
if RUN_FINAL_TEST:
    test_metrics = evaluate_classifier(model, X_test, y_test, "logistic_regression_test")
else:
    print("Testset niet geëvalueerd: leg eerst alle modelkeuzes vast.")

## 8. Volgende beslissing

Mogelijke vervolgstappen zijn het tunen van regularisatieparameter `C`, het vergelijken van L1- en L2-regularisatie en het kiezen van een waarschijnlijkheidsdrempel. Deze keuzes worden pas na overleg toegevoegd.